# Train DAVID-AI to chat in English (free Kaggle GPU)

**Before you start (once):**
1. Verify your phone number on Kaggle (needed for GPU and internet).
2. In this notebook's right panel: **Accelerator → GPU T4 x2**, and **Internet → On**.

**How to run:** click **Save Version → Save & Run All (Commit)**. It runs top to bottom in the background, even if you close the tab, and keeps everything in `/kaggle/working`. Kaggle stops a session at 12 hours, so the time limits below add up to about 10.5 hours.

When it finishes, open the version's **Output** tab and download `david-ai-trained.zip`.

**To continue training in a later session:** upload `model.safetensors` and `model_config.json` from that zip as a Kaggle dataset, add it to this notebook (**Add Input**), and run again. It picks them up automatically.

In [ ]:
# Settings: change these numbers if you like.
ENGLISH_STORIES = 600_000      # Stage 1 data: TinyStories (simple, fluent English)
CHAT_CONVERSATIONS = 150_000   # Stage 2 data: smol-smoltalk conversations
CODE_FILES = 3_000             # Python files mixed in so coding isn't forgotten (0 = skip)
ENGLISH_HOURS = 6.0            # Stage 1 time limit
CHAT_HOURS = 3.5               # Stage 2 time limit
START_FRESH = False            # True = ignore any uploaded model and start from random weights

In [ ]:
%cd /kaggle/working
!if [ -d DAVID-AI ]; then cd DAVID-AI && git pull; else git clone https://github.com/seundeeun-collab/DAVID-AI.git; fi
%cd /kaggle/working/DAVID-AI
# Kaggle already has a CUDA build of PyTorch. Don't install requirements.txt here: it pins the CPU build.
!pip install -q tokenizers safetensors
!python -c "import datasets" 2>/dev/null || pip install -q datasets
import torch
print("PyTorch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE. Turn on Accelerator -> GPU T4 x2, then run again.")

In [ ]:
# Continue from an uploaded model if one was added as input.
import glob, shutil, os
weights = glob.glob('/kaggle/input/**/model.safetensors', recursive=True)
configs = glob.glob('/kaggle/input/**/model_config.json', recursive=True)
if START_FRESH:
    for f in ('model.safetensors', 'model_config.json'):
        if os.path.exists(f): os.remove(f)
    print('Starting from random weights.')
elif weights and configs:
    shutil.copy(weights[0], 'model.safetensors'); shutil.copy(configs[0], 'model_config.json')
    print('Continuing from', weights[0])
else:
    print('No uploaded model found: starting from random weights.')

In [ ]:
# Download and tokenize the training data (about 20-40 minutes; cached if you re-run).
if CODE_FILES:
    !python download_data.py --count {CODE_FILES}
    !python prepare_data.py --code
!python prepare_data.py --english {ENGLISH_STORIES} --chat {CHAT_CONVERSATIONS}

In [ ]:
# Stage 1: learn English. Saves the best weights as it goes, and stops cleanly at the time limit.
!python train.py --stage english --max-hours {ENGLISH_HOURS} --workers 2

In [ ]:
# Stage 2: learn to chat. Continues from Stage 1 and only trains on the assistant's replies.
!python train.py --stage chat --max-hours {CHAT_HOURS} --workers 2

In [ ]:
# Test with the same fixed questions every time.
!python eval.py

In [ ]:
# Package the results to download.
!zip -j /kaggle/working/david-ai-trained.zip model.safetensors model_config.json model-english.safetensors model-chat.safetensors eval_results.json training.jsonl 2>/dev/null
!rm -f checkpoint-*-latest.safetensors
print('Download /kaggle/working/david-ai-trained.zip from the Output tab.')
print('On your PC: unzip model.safetensors and model_config.json into your DAVID-AI folder, then run: python app.py')